# Multimodal Reasoning for STEM

This notebook is designed to run on Google Colab with GPU support.

**Requirements:**
- Google Colab with GPU runtime (T4 or better)
- HuggingFace account (for model upload)

## 1. Setup

In [1]:
# Check GPU
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# Install dependencies
!pip install -q torch torchvision
!pip install -q transformers>=4.45.0 accelerate>=0.25.0
!pip install -q peft>=0.7.0 bitsandbytes>=0.41.0
!pip install -q datasets>=2.14.0
!pip install -q editdistance nltk sacrebleu
!pip install -q wandb python-dotenv

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)

True

In [3]:
# Clone the repository
!git clone https://github.com/dmitryz1024/Multimodal_Reasoning_for_STEM.git
%cd Multimodal_Reasoning_for_STEM

fatal: destination path 'Multimodal_Reasoning_for_STEM' already exists and is not an empty directory.
/content/Multimodal_Reasoning_for_STEM


In [ ]:
# Load environment variables from .env file
from dotenv import load_dotenv
import os
from google.colab import files

# Upload .env file if it doesn't exist
if not os.path.exists('.env'):
    print("Please upload your .env file containing WANDB_API_KEY")
    uploaded = files.upload()
    if '.env' in uploaded:
        print(".env file uploaded successfully")
    else:
        print("Warning: .env file not found. Please ensure WANDB_API_KEY is set.")

# Load environment variables
load_dotenv()

# Login to Weights & Biases
import wandb
wandb_api_key = os.getenv("WANDB_API_KEY")
if wandb_api_key:
    wandb.login(key=wandb_api_key)
    print("Successfully logged in to Weights & Biases")
else:
    print("Warning: WANDB_API_KEY not found in .env file. Wandb logging may not work.")

In [4]:
# Verify setup
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.10.0+cpu
CUDA: False


## 2. HuggingFace Login

In [5]:
from huggingface_hub import login, notebook_login
import os

# Try to login with token from environment
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(hf_token)
    print("Logged in using HF_TOKEN environment variable")
else:
    # Fallback to notebook login (for Colab)
    try:
        notebook_login()
    except Exception as e:
        print(f"Could not authenticate with Hugging Face: {e}")
        print("Please set HF_TOKEN environment variable or run in Colab")
        print("You can get a token from: https://huggingface.co/settings/tokens")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## 3. Load Datasets

In [6]:
from datasets import load_dataset

# Load primary dataset
print("Loading LaTeX_OCR dataset...")
ds_latex_ocr = load_dataset("linxy/LaTeX_OCR", "human_handwrite")
print(ds_latex_ocr)

# Load secondary dataset (sample)
print("\nLoading MathWriting dataset...")
ds_mathwriting = load_dataset("deepcopy/MathWriting-human", split="train")
ds_mathwriting = ds_mathwriting.shuffle(seed=42).select(range(10000))  # Sample 10k
print(f"MathWriting samples: {len(ds_mathwriting)}")

Loading LaTeX_OCR dataset...


README.md: 0.00B [00:00, ?B/s]

human_handwrite/train-00000-of-00001.par(…):   0%|          | 0.00/16.2M [00:00<?, ?B/s]

human_handwrite/validation-00000-of-0000(…):   0%|          | 0.00/961k [00:00<?, ?B/s]

human_handwrite/test-00000-of-00001.parq(…):   0%|          | 0.00/906k [00:00<?, ?B/s]

...

data/val-00000-of-00001-184984e66f80ed7a(…):   0%|          | 0.00/81.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/229864 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7644 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/15674 [00:00<?, ? examples/s]

MathWriting samples: 10000


## 4. Training Setup 1: LaTeX_OCR Only

In [7]:
import sys
sys.path.insert(0, 'src')

from src.train import TrainConfig, train

# Configuration
config = TrainConfig(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    primary_subset="human_handwrite",
    use_secondary=False,
    num_epochs=3,
    batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    use_lora=True,
    lora_r=16,
    lora_alpha=32,
    load_in_4bit=True,
    bf16=True,
)

print("Configuration:")
for k, v in config.__dict__.items():
    print(f"  {k}: {v}")

Configuration:
  model_name: HuggingFaceTB/SmolVLM-256M-Instruct
  use_lora: True
  lora_r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  load_in_4bit: True
  primary_dataset: linxy/LaTeX_OCR
  primary_subset: human_handwrite
  use_secondary: False
  secondary_sample_size: 10000
  output_dir: ./checkpoints
  num_epochs: 3
  batch_size: 4
  gradient_accumulation_steps: 4
  learning_rate: 0.0002
  weight_decay: 0.01
  warmup_ratio: 0.03
  max_grad_norm: 1.0
  bf16: True
  fp16: False
  gradient_checkpointing: True
  seed: 42
  logging_steps: 10
  save_strategy: epoch
  eval_strategy: epoch


In [ ]:
# Train with LaTeX_OCR only
model_latex_only, processor = train(
    config=config,
    run_name="sft_latex_ocr_only",
    wandb_project="handwritten-latex-ocr"
)

Starting training: sft_latex_ocr_only


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dmitryz1024 (dmitryz1024-hse) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loading model: HuggingFaceTB/SmolVLM-256M-Instruct


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Model loaded: Idefics3ForConditionalGeneration
Model parameters: 157,394,496
Preparing model for LoRA training...
Trainable parameters: 2,727,936 / 259,212,864 (1.05%)

Loading datasets...
Loading LaTeX_OCR dataset with subset: human_handwrite


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loaded dataset: DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 1200
    })
    validation: Dataset({
        features: ['image', 'text'],
        num_rows: 68
    })
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 70
    })
})

Train samples: 1200
Eval samples: 68

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2402: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


## 5. Training Setup 2: Combined Dataset

In [ ]:
# Update config for combined training
config.use_secondary = True
config.secondary_sample_size = 10000

# Train with combined dataset
model_combined, processor = train(
    config=config,
    run_name="sft_latex_ocr_mathwriting",
    wandb_project="handwritten-latex-ocr"
)

## 6. Evaluation

In [ ]:
from src.evaluate import run_full_evaluation, EvalConfig

eval_config = EvalConfig(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    subset="human_handwrite",
    num_samples=70
)

results = run_full_evaluation(
    base_model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    checkpoint_latex_ocr="./checkpoints/sft_latex_ocr_only/final",
    checkpoint_combined="./checkpoints/sft_latex_ocr_mathwriting/final",
    config=eval_config
)

# Save results
import json
with open('evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\nResults saved to evaluation_results.json")

## 7. Upload to HuggingFace Hub

In [ ]:
# Upload LaTeX_OCR only model
!python scripts/upload_to_hub.py \
    --local_path ./checkpoints/sft_latex_ocr_only/final \
    --repo_id YOUR_USERNAME/latex-ocr-smolvlm-latex-only \
    --training_setup "SFT with LaTeX_OCR only"

In [ ]:
# Upload combined model
!python scripts/upload_to_hub.py \
    --local_path ./checkpoints/sft_latex_ocr_mathwriting/final \
    --repo_id YOUR_USERNAME/latex-ocr-smolvlm-combined \
    --training_setup "SFT with LaTeX_OCR + MathWriting"

## 8. Test Inference

In [ ]:
from src.inference import LatexOCRInference
from PIL import Image
import matplotlib.pyplot as plt

# Load fine-tuned model
engine = LatexOCRInference(
    model_name="HuggingFaceTB/SmolVLM-256M-Instruct",
    adapter_path="./checkpoints/sft_latex_ocr_only/final"
)

# Test on a few examples
test_ds = ds_latex_ocr['test']

for i in range(5):
    example = test_ds[i]
    image = example['image']
    reference = example.get('text') or example.get('latex', '')
    
    prediction = engine.predict(image)
    
    print(f"\nExample {i+1}:")
    print(f"Reference:  {reference}")
    print(f"Prediction: {prediction}")
    
    plt.figure(figsize=(6, 2))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

## 9. Download Results

In [ ]:
# Download evaluation results and checkpoints
from google.colab import files

files.download('evaluation_results.json')

# Optionally zip and download checkpoints
!zip -r checkpoints.zip checkpoints/
files.download('checkpoints.zip')